# 02 — Scripts and files

The wasm module has an in-memory filesystem (the working directory is
`/work`). LAMMPS reads and writes it exactly like a real disk: data files,
potentials, dumps, logs. This notebook writes files in, reads results out, and
loads a LAMMPS script over HTTP.

In [ ]:
// Load lammps.js (served by this site under ./lammps/). Run this cell first.
// The site root is derived from wherever this code runs: the kernel iframe
// inherits the page URL ({site}/lab/…), the worker kernel lives under
// {site}/extensions/….
const base = globalThis.document?.baseURI ?? location.href;
globalThis.SITE ??= base.replace(/(extensions|lab|notebooks|files|tree|repl|consoles|edit)\/.*$/, "");
globalThis.LammpsClient ??= (await import(new URL("lammps/client.js", globalThis.SITE))).LammpsClient;
"lammps.js loaded ✓"

## `runInput`: write a file and run it

`runInput(path, script)` writes the script into the filesystem and tells
LAMMPS to run that file. The script below also dumps the final configuration
to `atoms.dump`.

In [ ]:
globalThis.lammps = await LammpsClient.create({ print: (line) => console.log(line) });
lammps.start();

lammps.runInput("melt.in", `
  units         lj
  atom_style    atomic
  lattice       fcc 0.8442
  region        box block 0 2 0 2 0 2
  create_box    1 box
  create_atoms  1 box
  mass          1 1.0
  pair_style    lj/cut 2.5
  pair_coeff    1 1 1.0 1.0 2.5
  write_dump    all atom atoms.dump
`);
console.log("files in workdir:", lammps.module.FS.readdir(".").join("  "));

## Read files back out

Anything LAMMPS wrote can be read through the Emscripten `FS` API — parse it,
plot it, or offer it as a download.

In [ ]:
const dump = lammps.module.FS.readFile("atoms.dump", { encoding: "utf8" });
console.log(dump.split("\n").slice(0, 9).join("\n"));
console.log("…");

## Write files in

`writeFile` puts any text or bytes into the filesystem — this is how you
provide data files, restart files, or potential files (e.g. fetched from the
web) before a run.

In [ ]:
lammps.writeFile("note.txt", "any bytes or text");
console.log("workdir:", lammps.module.FS.readdir(".").join("  "));
lammps.removeFile("note.txt");

## Load a script from the site

Notebook content is served over HTTP under `files/`, so tutorials can ship
inputs and data alongside the notebooks. This fetches a LAMMPS script and runs
it. The same pattern works for any URL — e.g. potential files from the LAMMPS
GitHub repository.

In [ ]:
const script = await (await fetch(new URL("files/data/lj-melt.in", globalThis.SITE))).text();
console.log("fetched script:\n" + script);
lammps.runCommand("clear");
lammps.runScript(script);
lammps.dispose();